# MediaPipe posture timeline notebook v2

Este notebook implementa a segunda versão do protótipo de triagem geométrica de postura. A mudança central é migrar para `MediaPipe Tasks HolisticLandmarker`, extraindo pose, face mesh e mãos no mesmo resultado.

As principais diferenças em relação ao v1 são:

- landmarks anatômicos completos para braços, punhos, quadris, face e mãos;
- fallback por `pose wrist/elbow` quando a mão do Holistic não aparece;
- regras concorrentes baseadas em `score`, sem supressão fixa entre rosto, pescoço e peito;
- JSON v2 com componentes de score, fonte de evidência e regras exibíveis;
- overlay com braços, mãos completas e faixa de pescoço para depuração visual.

Observação: o notebook não deve ser lido como detector clínico ou emocional. Ele apenas descreve padrões geométricos observáveis nos frames amostrados.


## 1) Instalação opcional do ambiente

Execute a próxima célula apenas se o ambiente ainda não tiver as dependências necessárias. Se você já estiver com `opencv-python`, `mediapipe`, `numpy`, `pandas` e `tqdm` instalados, pode pular esta etapa. Os modelos `.task` são resolvidos pelo notebook e o modelo de mãos é baixado automaticamente se estiver faltando.


In [1]:
# Rode esta célula uma vez por ambiente, se necessário.
%pip install -q opencv-python mediapipe numpy pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Imports e checagem do runtime

Esta etapa importa as bibliotecas usadas pelo pipeline v2 e mostra um resumo rápido do ambiente. A implementação depende da API nova de `mediapipe.tasks.vision.HolisticLandmarker`.


In [2]:
# Importa dependências e verifica se o runtime local possui a API Holistic Tasks necessária para o v2.
from __future__ import annotations

import json
import math
import urllib.request
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import cv2
import numpy as np
from IPython.display import Markdown, display
from tqdm.auto import tqdm

try:
    import mediapipe as mp
except ImportError:
    mp = None

try:
    import pandas as pd
except ImportError:
    pd = None

holistic_available = bool(mp is not None and hasattr(mp, 'tasks') and hasattr(mp.tasks.vision, 'HolisticLandmarker'))

display(Markdown('\n'.join([
    '### Runtime check',
    f'- OpenCV version: `{cv2.__version__}`',
    f'- NumPy version: `{np.__version__}`',
    f'- MediaPipe available: `{mp is not None}`',
    f'- HolisticLandmarker available: `{holistic_available}`',
    f'- Pandas available: `{pd is not None}`',
    f'- tqdm available: `{tqdm is not None}`',
])))

if not holistic_available:
    display(Markdown(
        '**MediaPipe HolisticLandmarker is missing.** Execute the installation cell above and rerun the notebook before processing videos.'
    ))


### Runtime check
- OpenCV version: `4.13.0`
- NumPy version: `2.4.4`
- MediaPipe available: `True`
- HolisticLandmarker available: `True`
- Pandas available: `True`
- tqdm available: `True`

## 3) Configuração do notebook

Aqui ficam caminhos, limiares, parâmetros de pescoço e o asset oficial do Holistic Tasks. A saída v2 usa pastas próprias para facilitar comparação com os artefatos gerados pelo notebook original.


In [3]:
# Define caminhos, limiares e parâmetros geométricos usados por todo o pipeline v2.

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_ROOT.parent if NOTEBOOK_ROOT.name == 'concepts_video' else NOTEBOOK_ROOT
ARTIFACTS_DIR = NOTEBOOK_ROOT / 'artifacts'
INPUT_VIDEO_DIR = NOTEBOOK_ROOT / 'data' / 'video'
OUTPUT_JSON_DIR = NOTEBOOK_ROOT / 'outputs' / 'posture_timelines_v2'
OUTPUT_VIDEO_DIR = NOTEBOOK_ROOT / 'outputs' / 'annotated_videos_v2'

HOLISTIC_LANDMARKER_MODEL_PATH = ARTIFACTS_DIR / 'holistic_landmarker.task'
HOLISTIC_LANDMARKER_MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/holistic_landmarker/'
    'holistic_landmarker/float16/1/holistic_landmarker.task'
)

SUPPORTED_VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.m4v'}

TARGET_SAMPLE_FPS = 5.0
VISIBILITY_THRESHOLD = 0.50
EMA_ALPHA = 0.35
EMA_MAX_GAP_FRAMES = 3
WINDOW_SECONDS = 8.0
WINDOW_STRIDE_SECONDS = 2.0
MIN_SHOULDER_WIDTH_NORM = 0.02

DISPLAY_SCORE_THRESHOLD = 0.35
RULE_SCORE_THRESHOLD = DISPLAY_SCORE_THRESHOLD
FRAME_STRONG_SCORE_THRESHOLD = 0.70

BASE_NECK_FRACTION = 0.20
MAX_NECK_TILT_BONUS = 0.25
MAX_NECK_FRACTION = 0.45
CHEST_OVERLAP_FRACTION = 0.08
HEAD_DROP_NEUTRAL_RATIO = 0.42
HEAD_DROP_STRONG_RATIO = 0.26

HAND_SOURCE_QUALITY = {
    'holistic_hand': 1.00,
    'pose_wrist_fallback': 0.75,
    'arm_proxy': 0.45,
    'unknown': 0.00,
}

PIPELINE_BACKEND = 'MediaPipe Tasks (HolisticLandmarker)'
SCHEMA_VERSION = 'posture_signal_timeline_v2'
COORDINATE_MODE_WORLD = 'mixed_world_pose_face_hands'
COORDINATE_MODE_2D = 'image_2d_only'

RULE_ORDER = [
    'hand_on_face',
    'hand_on_neck',
    'hand_on_chest',
    'head_down',
    'forward_head',
    'rounded_shoulders_or_asymmetry',
]

CONTACT_RULE_NAMES = {'hand_on_face', 'hand_on_neck', 'hand_on_chest'}

RULE_DISPLAY_NAMES = {
    'hand_on_face': 'hand-to-face contact',
    'hand_on_neck': 'hand-to-neck contact',
    'hand_on_chest': 'hand-to-chest contact',
    'head_down': 'head down / head tilt',
    'forward_head': 'forward head',
    'rounded_shoulders_or_asymmetry': 'upper-body tension / asymmetry',
}

FRAME_LEVEL_COLORS = {
    'ok': (80, 170, 80),
    'possible_signal': (0, 180, 255),
    'strong_signal': (0, 90, 255),
    'insufficient_data': (120, 120, 120),
}

WINDOW_LEVEL_SEVERITY = {
    'insufficient_data': -1,
    'ok': 0,
    'possible_signal': 1,
    'strong_signal': 2,
}

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_VIDEO_PATHS = []
if INPUT_VIDEO_DIR.exists():
    DEFAULT_VIDEO_PATHS = sorted(
        path for path in INPUT_VIDEO_DIR.iterdir() if path.suffix.lower() in SUPPORTED_VIDEO_EXTENSIONS
    )

display(Markdown('\n'.join([
    '### Current configuration v2',
    f'- Notebook root: `{NOTEBOOK_ROOT}`',
    f'- Input video dir: `{INPUT_VIDEO_DIR}`',
    f'- Holistic model path: `{HOLISTIC_LANDMARKER_MODEL_PATH}`',
    f'- JSON output dir: `{OUTPUT_JSON_DIR}`',
    f'- Annotated MP4 dir: `{OUTPUT_VIDEO_DIR}`',
    f'- Default videos found: `{len(DEFAULT_VIDEO_PATHS)}`',
    f'- Sample target FPS: `{TARGET_SAMPLE_FPS:.1f}`',
    f'- EMA alpha: `{EMA_ALPHA:.2f}`',
    f'- Rule/display score threshold: `{RULE_SCORE_THRESHOLD:.2f}`',
    f'- Neck fraction base/max: `{BASE_NECK_FRACTION:.2f} / {MAX_NECK_FRACTION:.2f}`',
    f'- Window size / stride: `{WINDOW_SECONDS:.1f}s / {WINDOW_STRIDE_SECONDS:.1f}s`',
])))


### Current configuration v2
- Notebook root: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video`
- Input video dir: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\data\video`
- Holistic model path: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\artifacts\holistic_landmarker.task`
- JSON output dir: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\posture_timelines_v2`
- Annotated MP4 dir: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\annotated_videos_v2`
- Default videos found: `4`
- Sample target FPS: `5.0`
- EMA alpha: `0.35`
- Rule/display score threshold: `0.35`
- Neck fraction base/max: `0.20 / 0.45`
- Window size / stride: `8.0s / 2.0s`

## 4) Download do modelo `.task`

Esta célula baixa explicitamente o asset oficial `holistic_landmarker.task` para `concepts_video/artifacts`. A URL usa a versão fixa `float16/1`, validada previamente, para manter reprodutibilidade.


In [4]:
# Baixa o asset oficial do Holistic Tasks para o disco antes de qualquer processamento de vídeo.
def download_file_with_progress(url: str, destination_path: Path) -> None:
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.urlopen(url)
    total_bytes = int(request.headers.get('Content-Length') or 0)
    temp_path = destination_path.with_suffix(destination_path.suffix + '.tmp')

    try:
        with request, temp_path.open('wb') as file_handle, tqdm(
            total=total_bytes or None,
            desc=f'Downloading {destination_path.name}',
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
        ) as progress_bar:
            while True:
                chunk = request.read(1024 * 1024)
                if not chunk:
                    break
                file_handle.write(chunk)
                progress_bar.update(len(chunk))
        temp_path.replace(destination_path)
    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise


# Garante que o modelo Holistic esteja presente; use `force_download=True` para substituir o arquivo local.
def download_required_model_assets(force_download: bool = False) -> dict[str, Path]:
    if HOLISTIC_LANDMARKER_MODEL_PATH.exists() and not force_download:
        return {'holistic': HOLISTIC_LANDMARKER_MODEL_PATH}

    download_file_with_progress(HOLISTIC_LANDMARKER_MODEL_URL, HOLISTIC_LANDMARKER_MODEL_PATH)
    return {'holistic': HOLISTIC_LANDMARKER_MODEL_PATH}


# Interrompe o processamento com uma mensagem clara se o asset não foi baixado.
def assert_model_assets_available() -> None:
    if not HOLISTIC_LANDMARKER_MODEL_PATH.exists():
        raise FileNotFoundError(
            'Missing model asset: holistic_landmarker.task. Run the model download cell before processing videos.'
        )


MODEL_ASSETS = download_required_model_assets(force_download=False)
display(Markdown('\n'.join([
    '### Model asset ready',
    f'- Holistic task: `{MODEL_ASSETS["holistic"]}`',
    f'- Source URL: `{HOLISTIC_LANDMARKER_MODEL_URL}`',
])))


### Model asset ready
- Holistic task: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\artifacts\holistic_landmarker.task`
- Source URL: `https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task`

## 5) Modelos de dados e extração de landmarks

O Holistic Landmarker devolve pose, face mesh e mãos no mesmo resultado. Nesta etapa nós extraímos os pontos necessários para regras, fallback e overlay, mantendo os nomes explícitos no código para facilitar auditoria.


In [5]:
# Define estruturas pequenas e extratores explícitos para pose, face mesh e mãos do Holistic.
@dataclass(frozen=True)
class PointData:
    x: float
    y: float
    z: float | None = None
    visibility: float | None = None
    space: str = 'image'


POSE_LANDMARK_INDEX = {
    'nose': 0,
    'left_eye_inner': 1,
    'left_eye': 2,
    'left_eye_outer': 3,
    'right_eye_inner': 4,
    'right_eye': 5,
    'right_eye_outer': 6,
    'left_ear': 7,
    'right_ear': 8,
    'mouth_left': 9,
    'mouth_right': 10,
    'left_shoulder': 11,
    'right_shoulder': 12,
    'left_elbow': 13,
    'right_elbow': 14,
    'left_wrist': 15,
    'right_wrist': 16,
    'left_pinky': 17,
    'right_pinky': 18,
    'left_index': 19,
    'right_index': 20,
    'left_thumb': 21,
    'right_thumb': 22,
    'left_hip': 23,
    'right_hip': 24,
}

HAND_LANDMARK_INDEX = {
    'wrist': 0,
    'thumb_cmc': 1,
    'thumb_mcp': 2,
    'thumb_ip': 3,
    'thumb_tip': 4,
    'index_mcp': 5,
    'index_pip': 6,
    'index_dip': 7,
    'index_tip': 8,
    'middle_mcp': 9,
    'middle_pip': 10,
    'middle_dip': 11,
    'middle_tip': 12,
    'ring_mcp': 13,
    'ring_pip': 14,
    'ring_dip': 15,
    'ring_tip': 16,
    'pinky_mcp': 17,
    'pinky_pip': 18,
    'pinky_dip': 19,
    'pinky_tip': 20,
}

FACE_LANDMARK_INDEX = {
    'face_forehead': 10,
    'face_nose_tip': 1,
    'face_nose_bridge': 6,
    'face_chin': 152,
    'face_left_eye_outer': 33,
    'face_left_eye_inner': 133,
    'face_right_eye_inner': 362,
    'face_right_eye_outer': 263,
    'face_mouth_left': 61,
    'face_mouth_right': 291,
    'face_mouth_upper': 13,
    'face_mouth_lower': 14,
    'face_lower_lip': 17,
    'face_jaw_left': 234,
    'face_jaw_right': 454,
    'face_cheek_left': 205,
    'face_cheek_right': 425,
}

FACE_REGION_NAMES = [
    'face_nose_tip',
    'face_nose_bridge',
    'face_chin',
    'face_mouth_left',
    'face_mouth_right',
    'face_mouth_upper',
    'face_mouth_lower',
    'face_lower_lip',
    'face_jaw_left',
    'face_jaw_right',
    'face_cheek_left',
    'face_cheek_right',
    'face_left_eye_outer',
    'face_right_eye_outer',
]

FACE_PITCH_PNP_NAMES = [
    'face_nose_tip',
    'face_chin',
    'face_left_eye_outer',
    'face_right_eye_outer',
    'face_mouth_left',
    'face_mouth_right',
]


# Converte um landmark bruto do MediaPipe para uma estrutura estável e serializável.
def make_point(landmark: Any, *, space: str, visibility: float | None = None) -> PointData:
    landmark_visibility = getattr(landmark, 'visibility', visibility)
    return PointData(
        x=float(landmark.x),
        y=float(landmark.y),
        z=float(getattr(landmark, 'z', 0.0)),
        visibility=float(landmark_visibility) if landmark_visibility is not None else None,
        space=space,
    )


# Normaliza containers diferentes da API Tasks para uma lista simples de landmarks.
def normalize_landmark_container(raw_landmarks: Any) -> list[Any]:
    if raw_landmarks is None:
        return []
    if hasattr(raw_landmarks, 'landmark'):
        raw_landmarks = getattr(raw_landmarks, 'landmark')
    if not raw_landmarks:
        return []

    first_item = raw_landmarks[0]
    if hasattr(first_item, 'x'):
        return list(raw_landmarks)
    if hasattr(first_item, 'landmark'):
        return list(first_item.landmark)
    if isinstance(first_item, (list, tuple)):
        return list(first_item)
    return list(raw_landmarks)


# Extrai landmarks nomeados e aplica filtro de visibilidade quando esse dado existe.
def landmark_list_to_points(
    landmark_list: Any,
    name_to_index: Mapping[str, int],
    *,
    space: str,
    require_visibility: bool = False,
    visibility_threshold: float = VISIBILITY_THRESHOLD,
) -> dict[str, PointData]:
    landmarks = normalize_landmark_container(landmark_list)
    points: dict[str, PointData] = {}
    for name, index in name_to_index.items():
        if index >= len(landmarks):
            continue
        landmark = landmarks[index]
        visibility = getattr(landmark, 'visibility', None)
        if require_visibility and visibility is not None and visibility < visibility_threshold:
            continue
        points[name] = make_point(landmark, space=space, visibility=visibility)
    return points


# Lê o resultado do HolisticLandmarker e organiza pose, face e mãos por grupo.
def extract_landmark_sets(holistic_result: Any) -> dict[str, dict[str, PointData]]:
    if holistic_result is None:
        return build_empty_landmark_sets()

    pose_landmarks = getattr(holistic_result, 'pose_landmarks', None)
    pose_world_landmarks = getattr(holistic_result, 'pose_world_landmarks', None)
    face_landmarks = getattr(holistic_result, 'face_landmarks', None)
    left_hand_landmarks = getattr(holistic_result, 'left_hand_landmarks', None)
    right_hand_landmarks = getattr(holistic_result, 'right_hand_landmarks', None)
    left_hand_world_landmarks = getattr(holistic_result, 'left_hand_world_landmarks', None)
    right_hand_world_landmarks = getattr(holistic_result, 'right_hand_world_landmarks', None)

    return {
        'pose_2d': landmark_list_to_points(
            pose_landmarks,
            POSE_LANDMARK_INDEX,
            space='image',
            require_visibility=True,
        ),
        'pose_world': landmark_list_to_points(
            pose_world_landmarks,
            POSE_LANDMARK_INDEX,
            space='world',
            require_visibility=False,
        ),
        'face_2d': landmark_list_to_points(
            face_landmarks,
            FACE_LANDMARK_INDEX,
            space='image',
            require_visibility=False,
        ),
        'left_hand_2d': landmark_list_to_points(
            left_hand_landmarks,
            HAND_LANDMARK_INDEX,
            space='image',
            require_visibility=False,
        ),
        'right_hand_2d': landmark_list_to_points(
            right_hand_landmarks,
            HAND_LANDMARK_INDEX,
            space='image',
            require_visibility=False,
        ),
        'left_hand_world': landmark_list_to_points(
            left_hand_world_landmarks,
            HAND_LANDMARK_INDEX,
            space='world',
            require_visibility=False,
        ),
        'right_hand_world': landmark_list_to_points(
            right_hand_world_landmarks,
            HAND_LANDMARK_INDEX,
            space='world',
            require_visibility=False,
        ),
    }


# Devolve a mesma estrutura do extrator, mas sem pontos, quando a inferência do frame falha.
def build_empty_landmark_sets() -> dict[str, dict[str, PointData]]:
    return {
        'pose_2d': {},
        'pose_world': {},
        'face_2d': {},
        'left_hand_2d': {},
        'right_hand_2d': {},
        'left_hand_world': {},
        'right_hand_world': {},
    }


# Copia apenas pontos 2D necessários para o overlay, evitando carregar dados world no renderizador.
def clone_drawing_points(landmark_sets: Mapping[str, dict[str, PointData]]) -> dict[str, dict[str, PointData]]:
    return {
        'pose_2d': dict(landmark_sets.get('pose_2d', {})),
        'face_2d': dict(landmark_sets.get('face_2d', {})),
        'left_hand_2d': dict(landmark_sets.get('left_hand_2d', {})),
        'right_hand_2d': dict(landmark_sets.get('right_hand_2d', {})),
    }


## 6) Suavização temporal com EMA

Landmarks podem oscilar entre frames. A média móvel exponencial reduz ruído preservando mudanças rápidas. Cada grupo de landmarks tem seu próprio suavizador para não misturar pose, face e mãos.


In [6]:
# Aplica suavização temporal independente para cada grupo de landmarks do Holistic.
def blend_optional(current_value: float | None, previous_value: float | None, alpha: float) -> float | None:
    if current_value is None and previous_value is None:
        return None
    if current_value is None:
        return previous_value
    if previous_value is None:
        return current_value
    return (alpha * current_value) + ((1.0 - alpha) * previous_value)


class EMASmoother:
    # Guarda o último valor suavizado de cada landmark e reseta quando o ponto some por muitos frames.
    def __init__(self, alpha: float = EMA_ALPHA, max_gap_frames: int = EMA_MAX_GAP_FRAMES) -> None:
        self.alpha = alpha
        self.max_gap_frames = max_gap_frames
        self.state: dict[str, dict[str, Any]] = {}

    # Atualiza um conjunto de pontos e devolve a versão suavizada do frame atual.
    def update(self, frame_idx: int, points: Mapping[str, PointData]) -> dict[str, PointData]:
        smoothed: dict[str, PointData] = {}
        for name, point in points.items():
            previous = self.state.get(name)
            if previous is None or (frame_idx - previous['frame_idx']) > self.max_gap_frames:
                smoothed_point = point
            else:
                previous_point = previous['point']
                smoothed_point = PointData(
                    x=(self.alpha * point.x) + ((1.0 - self.alpha) * previous_point.x),
                    y=(self.alpha * point.y) + ((1.0 - self.alpha) * previous_point.y),
                    z=blend_optional(point.z, previous_point.z, self.alpha),
                    visibility=point.visibility,
                    space=point.space,
                )

            self.state[name] = {'frame_idx': frame_idx, 'point': smoothed_point}
            smoothed[name] = smoothed_point
        return smoothed


# Cria um suavizador por grupo para preservar identidades e escalas diferentes.
def build_smoothers() -> dict[str, EMASmoother]:
    return {
        'pose_2d': EMASmoother(),
        'pose_world': EMASmoother(),
        'face_2d': EMASmoother(),
        'left_hand_2d': EMASmoother(),
        'right_hand_2d': EMASmoother(),
        'left_hand_world': EMASmoother(),
        'right_hand_world': EMASmoother(),
    }


# Aplica todos os suavizadores esperados e mantém chaves vazias para grupos ausentes.
def smooth_landmark_sets(
    frame_idx: int,
    raw_sets: Mapping[str, dict[str, PointData]],
    smoothers: Mapping[str, EMASmoother],
) -> dict[str, dict[str, PointData]]:
    return {
        name: smoothers[name].update(frame_idx, raw_sets.get(name, {}))
        for name in [
            'pose_2d',
            'pose_world',
            'face_2d',
            'left_hand_2d',
            'right_hand_2d',
            'left_hand_world',
            'right_hand_world',
        ]
    }


## 7) Geometria de referência e evidências

Esta etapa constrói as referências usadas pelas regras: largura dos ombros, faixa dinâmica de pescoço, centro do peito, score composto de inclinação da cabeça e fontes de evidência por lado.


In [7]:
# Calcula referências geométricas, score de inclinação e fontes de evidência para as regras.
def optional_round(value: float | None, digits: int = 4) -> float | None:
    if value is None:
        return None
    return round(float(value), digits)


def clamp_value(value: float, low: float = 0.0, high: float = 1.0) -> float:
    return max(low, min(high, float(value)))


def point_distance(point_a: PointData | None, point_b: PointData | None) -> float | None:
    if point_a is None or point_b is None:
        return None
    if point_a.space == point_b.space == 'world' and point_a.z is not None and point_b.z is not None:
        return math.dist((point_a.x, point_a.y, point_a.z), (point_b.x, point_b.y, point_b.z))
    return math.dist((point_a.x, point_a.y), (point_b.x, point_b.y))


def midpoint(point_a: PointData | None, point_b: PointData | None) -> PointData | None:
    if point_a is None or point_b is None:
        return None
    z_value = None
    if point_a.z is not None and point_b.z is not None:
        z_value = (point_a.z + point_b.z) / 2.0
    visibility = None
    if point_a.visibility is not None and point_b.visibility is not None:
        visibility = min(point_a.visibility, point_b.visibility)
    return PointData(
        x=(point_a.x + point_b.x) / 2.0,
        y=(point_a.y + point_b.y) / 2.0,
        z=z_value,
        visibility=visibility,
        space=point_a.space,
    )


def offset_point(point: PointData | None, dx: float = 0.0, dy: float = 0.0) -> PointData | None:
    if point is None:
        return None
    return PointData(
        x=point.x + dx,
        y=point.y + dy,
        z=point.z,
        visibility=point.visibility,
        space=point.space,
    )


def centroid(points: Iterable[PointData | None], *, fallback_space: str = 'image') -> PointData | None:
    valid = [point for point in points if point is not None]
    if not valid:
        return None
    z_values = [point.z for point in valid if point.z is not None]
    visibility_values = [point.visibility for point in valid if point.visibility is not None]
    return PointData(
        x=float(np.mean([point.x for point in valid])),
        y=float(np.mean([point.y for point in valid])),
        z=float(np.mean(z_values)) if z_values else None,
        visibility=float(min(visibility_values)) if visibility_values else None,
        space=valid[0].space if valid else fallback_space,
    )


def valid_points(points: Iterable[PointData | None]) -> list[PointData]:
    return [point for point in points if point is not None]


def min_distance_to_named_points(
    point: PointData,
    named_points: Iterable[tuple[str, PointData]],
) -> tuple[float | None, str | None]:
    candidates = [
        (distance, name)
        for name, target in named_points
        for distance in [point_distance(point, target)]
        if distance is not None
    ]
    if not candidates:
        return None, None
    return min(candidates, key=lambda item: item[0])


def score_ratio_below(ratio: float | None, strong_ratio: float, weak_ratio: float) -> float:
    if ratio is None:
        return 0.0
    if ratio <= strong_ratio:
        return 1.0
    if ratio >= weak_ratio:
        return 0.0
    return 1.0 - ((ratio - strong_ratio) / max(weak_ratio - strong_ratio, 1e-6))


def score_ratio_above(ratio: float | None, weak_ratio: float, strong_ratio: float) -> float:
    if ratio is None:
        return 0.0
    if ratio <= weak_ratio:
        return 0.0
    if ratio >= strong_ratio:
        return 1.0
    return (ratio - weak_ratio) / max(strong_ratio - weak_ratio, 1e-6)


def weighted_mean_score(components: Iterable[tuple[float | None, float]]) -> float | None:
    available = [(score, weight) for score, weight in components if score is not None and weight > 0]
    if not available:
        return None
    total_weight = sum(weight for _, weight in available)
    return sum(float(score) * weight for score, weight in available) / total_weight


# Estima pitch absoluto por solvePnP usando landmarks explícitos da face mesh.
def estimate_face_pitch_degrees(face_2d: Mapping[str, PointData]) -> float | None:
    if any(name not in face_2d for name in FACE_PITCH_PNP_NAMES):
        return None

    model_points = np.array([
        (0.0, 0.0, 0.0),
        (0.0, -63.6, -12.5),
        (-43.3, 32.7, -26.0),
        (43.3, 32.7, -26.0),
        (-28.9, -28.9, -24.1),
        (28.9, -28.9, -24.1),
    ], dtype=np.float64)
    image_points = np.array(
        [[face_2d[name].x, face_2d[name].y] for name in FACE_PITCH_PNP_NAMES],
        dtype=np.float64,
    )
    camera_matrix = np.array(
        [[1.0, 0.0, 0.5], [0.0, 1.0, 0.5], [0.0, 0.0, 1.0]],
        dtype=np.float64,
    )
    distortion = np.zeros((4, 1), dtype=np.float64)

    try:
        success, rotation_vector, _ = cv2.solvePnP(
            model_points,
            image_points,
            camera_matrix,
            distortion,
            flags=cv2.SOLVEPNP_ITERATIVE,
        )
        if not success:
            return None
        rotation_matrix, _ = cv2.Rodrigues(rotation_vector)
        decomposition = cv2.RQDecomp3x3(rotation_matrix)
        angles = decomposition[0]
        return float(angles[0])
    except Exception:
        return None


# Combina pitch de face, altura ombros-nariz e compressão ombro-orelha sem deixar uma pista isolada dominar.
def compute_head_tilt_components(
    pose_2d: Mapping[str, PointData],
    face_2d: Mapping[str, PointData],
    shoulder_width: float | None,
    head_height_ratio: float | None,
) -> dict[str, float | str | None]:
    face_pitch_degrees = estimate_face_pitch_degrees(face_2d)
    face_pitch_score = None
    if face_pitch_degrees is not None and math.isfinite(face_pitch_degrees):
        face_pitch_score = clamp_value((abs(face_pitch_degrees) - 10.0) / 25.0)

    head_drop_score = None
    if head_height_ratio is not None:
        head_drop_score = clamp_value(
            (HEAD_DROP_NEUTRAL_RATIO - head_height_ratio)
            / max(HEAD_DROP_NEUTRAL_RATIO - HEAD_DROP_STRONG_RATIO, 1e-6)
        )

    shoulder_ear_compression_score = None
    if shoulder_width is not None and shoulder_width > 0:
        left_ratio = point_distance(pose_2d.get('left_shoulder'), pose_2d.get('left_ear'))
        right_ratio = point_distance(pose_2d.get('right_shoulder'), pose_2d.get('right_ear'))
        ratios = [
            ratio / shoulder_width
            for ratio in [left_ratio, right_ratio]
            if ratio is not None
        ]
        if ratios:
            shoulder_ear_compression_score = score_ratio_below(float(np.mean(ratios)), 0.30, 0.45)

    head_tilt_score = weighted_mean_score([
        (face_pitch_score, 0.65),
        (head_drop_score, 0.20),
        (shoulder_ear_compression_score, 0.15),
    ])
    if head_tilt_score is not None and face_pitch_score is None:
        head_tilt_score = min(head_tilt_score, 0.45)

    return {
        'face_pitch_degrees': face_pitch_degrees,
        'face_pitch_score': face_pitch_score,
        'head_drop_score': head_drop_score,
        'shoulder_ear_compression_score': shoulder_ear_compression_score,
        'head_tilt_score': head_tilt_score,
        'head_tilt_basis': 'face_pose_composite' if face_pitch_score is not None else 'pose_only_capped',
    }


def estimate_arm_proxy_point(shoulder: PointData | None, elbow: PointData | None) -> PointData | None:
    if shoulder is None or elbow is None:
        return None
    dx = elbow.x - shoulder.x
    dy = elbow.y - shoulder.y
    dz = None
    if shoulder.z is not None and elbow.z is not None:
        dz = elbow.z + ((elbow.z - shoulder.z) * 0.70)
    return PointData(
        x=clamp_value(elbow.x + (dx * 0.70), -0.25, 1.25),
        y=clamp_value(elbow.y + (dy * 0.70), -0.25, 1.25),
        z=dz,
        visibility=elbow.visibility,
        space=elbow.space,
    )


def build_side_contact_evidence(
    side: str,
    pose_2d: Mapping[str, PointData],
    hand_2d: Mapping[str, PointData],
) -> dict[str, Any]:
    if hand_2d:
        return {
            'side': side,
            'source': 'holistic_hand',
            'quality': HAND_SOURCE_QUALITY['holistic_hand'],
            'named_points': [(f'{side}_hand_2d.{name}', point) for name, point in hand_2d.items()],
            'landmarks_used': [f'{side}_hand_2d.{name}' for name in hand_2d],
            'notes': [],
        }

    wrist = pose_2d.get(f'{side}_wrist')
    if wrist is not None:
        return {
            'side': side,
            'source': 'pose_wrist_fallback',
            'quality': HAND_SOURCE_QUALITY['pose_wrist_fallback'],
            'named_points': [(f'pose_2d.{side}_wrist', wrist)],
            'landmarks_used': [f'pose_2d.{side}_wrist'],
            'notes': ['Hand landmarks missing; using pose wrist fallback.'],
        }

    shoulder = pose_2d.get(f'{side}_shoulder')
    elbow = pose_2d.get(f'{side}_elbow')
    proxy_point = estimate_arm_proxy_point(shoulder, elbow)
    if elbow is not None and proxy_point is not None:
        return {
            'side': side,
            'source': 'arm_proxy',
            'quality': HAND_SOURCE_QUALITY['arm_proxy'],
            'named_points': [
                (f'pose_2d.{side}_elbow', elbow),
                (f'pose_2d.{side}_estimated_wrist', proxy_point),
            ],
            'landmarks_used': [f'pose_2d.{side}_elbow', f'pose_2d.{side}_estimated_wrist'],
            'notes': ['Hand and wrist missing; using low-confidence arm proxy.'],
        }

    return {
        'side': side,
        'source': 'unknown',
        'quality': HAND_SOURCE_QUALITY['unknown'],
        'named_points': [],
        'landmarks_used': [],
        'notes': ['No usable hand, wrist or elbow evidence for this side.'],
    }


# Monta todas as referências que as regras precisam sem decidir uma classe única para o frame.
def compute_reference_geometry(landmark_sets: Mapping[str, dict[str, PointData]]) -> dict[str, Any]:
    pose_2d = landmark_sets.get('pose_2d', {})
    pose_world = landmark_sets.get('pose_world', {})
    face_2d = landmark_sets.get('face_2d', {})
    left_hand_2d = landmark_sets.get('left_hand_2d', {})
    right_hand_2d = landmark_sets.get('right_hand_2d', {})

    left_shoulder = pose_2d.get('left_shoulder')
    right_shoulder = pose_2d.get('right_shoulder')
    shoulder_mid = midpoint(left_shoulder, right_shoulder)
    shoulder_width = point_distance(left_shoulder, right_shoulder)
    shoulder_reference_ok = shoulder_width is not None and shoulder_width >= MIN_SHOULDER_WIDTH_NORM

    nose_2d = pose_2d.get('nose') or face_2d.get('face_nose_tip')
    shoulder_to_nose = None
    head_height_ratio = None
    if shoulder_reference_ok and shoulder_mid is not None and nose_2d is not None and shoulder_width:
        shoulder_to_nose = max(shoulder_mid.y - nose_2d.y, shoulder_width * 0.10)
        head_height_ratio = shoulder_to_nose / shoulder_width

    head_tilt_components = compute_head_tilt_components(pose_2d, face_2d, shoulder_width, head_height_ratio)
    head_tilt_score = head_tilt_components.get('head_tilt_score')
    neck_fraction = None
    neck_top_y = None
    neck_bottom_y = None
    neck_center = None
    if shoulder_reference_ok and shoulder_mid is not None and shoulder_to_nose is not None and shoulder_width:
        neck_fraction = clamp_value(
            BASE_NECK_FRACTION + (MAX_NECK_TILT_BONUS * float(head_tilt_score or 0.0)),
            BASE_NECK_FRACTION,
            MAX_NECK_FRACTION,
        )
        neck_top_y = shoulder_mid.y - (neck_fraction * shoulder_to_nose)
        neck_bottom_y = shoulder_mid.y + (CHEST_OVERLAP_FRACTION * shoulder_width)
        neck_center = PointData(
            x=shoulder_mid.x,
            y=(neck_top_y + neck_bottom_y) / 2.0,
            z=shoulder_mid.z,
            visibility=shoulder_mid.visibility,
            space='image',
        )

    hip_mid = midpoint(pose_2d.get('left_hip'), pose_2d.get('right_hip'))
    upper_chest_center = None
    if shoulder_mid is not None and hip_mid is not None:
        upper_chest_center = PointData(
            x=(shoulder_mid.x * 0.70) + (hip_mid.x * 0.30),
            y=(shoulder_mid.y * 0.70) + (hip_mid.y * 0.30),
            z=None,
            visibility=shoulder_mid.visibility,
            space='image',
        )
    elif shoulder_reference_ok and shoulder_mid is not None and shoulder_width:
        upper_chest_center = offset_point(shoulder_mid, dy=(0.22 * shoulder_width))

    face_region_points_named = [
        (name, face_2d[name])
        for name in FACE_REGION_NAMES
        if name in face_2d
    ]
    if not face_region_points_named:
        fallback_face_names = ['nose', 'mouth_left', 'mouth_right', 'left_ear', 'right_ear']
        face_region_points_named = [
            (f'pose_2d.{name}', pose_2d[name])
            for name in fallback_face_names
            if name in pose_2d
        ]
    face_points = [point for _, point in face_region_points_named]
    face_center = centroid(face_points)

    left_evidence = build_side_contact_evidence('left', pose_2d, left_hand_2d)
    right_evidence = build_side_contact_evidence('right', pose_2d, right_hand_2d)
    contact_evidence = [
        evidence for evidence in [left_evidence, right_evidence]
        if evidence.get('source') != 'unknown' and evidence.get('named_points')
    ]

    world_left_shoulder = pose_world.get('left_shoulder')
    world_right_shoulder = pose_world.get('right_shoulder')
    shoulder_mid_world = midpoint(world_left_shoulder, world_right_shoulder)
    shoulder_width_world = point_distance(world_left_shoulder, world_right_shoulder)
    nose_world = pose_world.get('nose')
    forward_head_world_ratio = None
    if shoulder_mid_world is not None and shoulder_width_world and shoulder_width_world > 0 and nose_world is not None:
        forward_head_world_ratio = (shoulder_mid_world.z - nose_world.z) / shoulder_width_world

    left_shoulder_ear_ratio = None
    right_shoulder_ear_ratio = None
    if shoulder_reference_ok and shoulder_width:
        left_shoulder_ear = point_distance(left_shoulder, pose_2d.get('left_ear'))
        right_shoulder_ear = point_distance(right_shoulder, pose_2d.get('right_ear'))
        if left_shoulder_ear is not None:
            left_shoulder_ear_ratio = left_shoulder_ear / shoulder_width
        if right_shoulder_ear is not None:
            right_shoulder_ear_ratio = right_shoulder_ear / shoulder_width

    shoulder_asymmetry_ratio = None
    if shoulder_reference_ok and left_shoulder is not None and right_shoulder is not None and shoulder_width:
        shoulder_asymmetry_ratio = abs(left_shoulder.y - right_shoulder.y) / shoulder_width

    coordinate_mode = COORDINATE_MODE_WORLD if forward_head_world_ratio is not None else COORDINATE_MODE_2D

    return {
        'pose_2d': pose_2d,
        'pose_world': pose_world,
        'face_2d': face_2d,
        'left_hand_2d': left_hand_2d,
        'right_hand_2d': right_hand_2d,
        'shoulder_mid': shoulder_mid,
        'shoulder_width': shoulder_width,
        'shoulder_reference_ok': shoulder_reference_ok,
        'shoulder_to_nose': shoulder_to_nose,
        'head_height_ratio': head_height_ratio,
        'head_tilt_components': head_tilt_components,
        'head_tilt_score': head_tilt_score,
        'neck_fraction': neck_fraction,
        'neck_top_y': neck_top_y,
        'neck_bottom_y': neck_bottom_y,
        'neck_center': neck_center,
        'upper_chest_center': upper_chest_center,
        'face_points': face_points,
        'face_region_points_named': face_region_points_named,
        'face_center': face_center,
        'face_anchor_count': len(face_points),
        'contact_evidence': contact_evidence,
        'left_evidence': left_evidence,
        'right_evidence': right_evidence,
        'hands_visible': bool(left_hand_2d or right_hand_2d),
        'left_hand_visible': bool(left_hand_2d),
        'right_hand_visible': bool(right_hand_2d),
        'left_wrist_visible': pose_2d.get('left_wrist') is not None,
        'right_wrist_visible': pose_2d.get('right_wrist') is not None,
        'left_elbow_visible': pose_2d.get('left_elbow') is not None,
        'right_elbow_visible': pose_2d.get('right_elbow') is not None,
        'left_shoulder_ear_ratio': left_shoulder_ear_ratio,
        'right_shoulder_ear_ratio': right_shoulder_ear_ratio,
        'shoulder_asymmetry_ratio': shoulder_asymmetry_ratio,
        'forward_head_world_ratio': forward_head_world_ratio,
        'coordinate_mode': coordinate_mode,
    }


## 8) Regras concorrentes por score

As regras `hand_on_face`, `hand_on_neck` e `hand_on_chest` agora são calculadas independentemente. Cada uma grava `score`, componentes, fonte de evidência e limiar aplicado.


In [8]:
# Avalia regras independentes por score, preservando componentes e fonte de evidência no JSON.
def serialize_metric_value(value: Any) -> Any:
    if isinstance(value, float):
        return optional_round(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {key: serialize_metric_value(item) for key, item in value.items()}
    if isinstance(value, list):
        return [serialize_metric_value(item) for item in value]
    return value


def build_scored_rule_result(
    *,
    score: float | None,
    evaluable: bool,
    raw_distance_ratio: float | None = None,
    zone_score: float | None = None,
    evidence_quality: float | None = None,
    source: str | None = None,
    closest_side: str | None = None,
    landmarks_used: list[str] | None = None,
    score_components: Mapping[str, Any] | None = None,
    notes: list[str] | None = None,
) -> dict[str, Any]:
    safe_score = clamp_value(score or 0.0)
    passed_threshold = bool(evaluable and safe_score >= RULE_SCORE_THRESHOLD)
    if not evaluable:
        state = 'unknown'
        strength = 'none'
    elif passed_threshold:
        state = 'true'
        strength = 'strong' if safe_score >= FRAME_STRONG_SCORE_THRESHOLD else 'weak'
    else:
        state = 'false'
        strength = 'none'

    return {
        'state': state,
        'strength': strength,
        'score': optional_round(safe_score),
        'passed_threshold': passed_threshold,
        'threshold': optional_round(RULE_SCORE_THRESHOLD),
        'raw_distance_ratio': optional_round(raw_distance_ratio),
        'zone_score': optional_round(zone_score),
        'evidence_quality': optional_round(evidence_quality),
        'source': source,
        'closest_side': closest_side,
        'landmarks_used': landmarks_used or [],
        'score_components': serialize_metric_value(dict(score_components or {})),
        'notes': notes or [],
    }


def face_vertical_score(point: PointData, geometry: Mapping[str, Any]) -> float:
    neck_top_y = geometry.get('neck_top_y')
    shoulder_width = geometry.get('shoulder_width') or 0.0
    if neck_top_y is None or shoulder_width <= 0:
        return 1.0
    if point.y <= neck_top_y:
        return 1.0
    decay_span = max(shoulder_width * 0.42, 1e-6)
    return clamp_value(1.0 - ((point.y - neck_top_y) / decay_span), 0.15, 1.0)


def neck_zone_score(point: PointData, geometry: Mapping[str, Any]) -> float:
    shoulder_mid = geometry.get('shoulder_mid')
    shoulder_width = geometry.get('shoulder_width') or 0.0
    neck_top_y = geometry.get('neck_top_y')
    neck_bottom_y = geometry.get('neck_bottom_y')
    if shoulder_mid is None or shoulder_width <= 0 or neck_top_y is None or neck_bottom_y is None:
        return 0.0

    if neck_top_y <= point.y <= neck_bottom_y:
        vertical_score = 1.0
    elif point.y < neck_top_y:
        vertical_score = clamp_value(1.0 - ((neck_top_y - point.y) / max(shoulder_width * 0.30, 1e-6)))
    else:
        vertical_score = clamp_value(1.0 - ((point.y - neck_bottom_y) / max(shoulder_width * 0.28, 1e-6)))

    horizontal_score = clamp_value(1.0 - (abs(point.x - shoulder_mid.x) / max(shoulder_width * 0.70, 1e-6)))
    return vertical_score * horizontal_score


def chest_vertical_score(point: PointData, geometry: Mapping[str, Any]) -> float:
    neck_bottom_y = geometry.get('neck_bottom_y')
    shoulder_width = geometry.get('shoulder_width') or 0.0
    if neck_bottom_y is None or shoulder_width <= 0:
        return 1.0
    if point.y >= neck_bottom_y:
        return 1.0
    return clamp_value(1.0 - ((neck_bottom_y - point.y) / max(shoulder_width * 0.35, 1e-6)))


def score_contact_point(rule_name: str, point: PointData, geometry: Mapping[str, Any]) -> dict[str, Any] | None:
    shoulder_width = geometry.get('shoulder_width')
    if not geometry.get('shoulder_reference_ok') or not shoulder_width:
        return None

    if rule_name == 'hand_on_face':
        target_distance, target_name = min_distance_to_named_points(point, geometry.get('face_region_points_named', []))
        if target_distance is None:
            return None
        raw_ratio = target_distance / shoulder_width
        distance_score = score_ratio_below(raw_ratio, 0.18, 0.36)
        vertical_score = face_vertical_score(point, geometry)
        score = distance_score * vertical_score
        return {
            'score': score,
            'raw_distance_ratio': raw_ratio,
            'zone_score': vertical_score,
            'target_name': target_name,
            'components': {
                'distance_score': distance_score,
                'vertical_score': vertical_score,
            },
        }

    if rule_name == 'hand_on_neck':
        neck_center = geometry.get('neck_center')
        center_distance = point_distance(point, neck_center)
        if center_distance is None:
            return None
        raw_ratio = center_distance / shoulder_width
        distance_score = score_ratio_below(raw_ratio, 0.10, 0.32)
        zone_score = neck_zone_score(point, geometry)
        score = (0.35 * distance_score) + (0.65 * zone_score)
        return {
            'score': score,
            'raw_distance_ratio': raw_ratio,
            'zone_score': zone_score,
            'target_name': 'dynamic_neck_zone',
            'components': {
                'distance_score': distance_score,
                'zone_score': zone_score,
                'neck_fraction': geometry.get('neck_fraction'),
                'head_tilt_score': geometry.get('head_tilt_score'),
            },
        }

    if rule_name == 'hand_on_chest':
        chest_center = geometry.get('upper_chest_center')
        center_distance = point_distance(point, chest_center)
        if center_distance is None:
            return None
        raw_ratio = center_distance / shoulder_width
        distance_score = score_ratio_below(raw_ratio, 0.12, 0.36)
        vertical_score = chest_vertical_score(point, geometry)
        score = distance_score * vertical_score
        return {
            'score': score,
            'raw_distance_ratio': raw_ratio,
            'zone_score': vertical_score,
            'target_name': 'upper_chest_center',
            'components': {
                'distance_score': distance_score,
                'vertical_score': vertical_score,
            },
        }

    return None


def evaluate_contact_rule(rule_name: str, geometry: Mapping[str, Any]) -> dict[str, Any]:
    if not geometry.get('shoulder_reference_ok'):
        return build_scored_rule_result(
            score=None,
            evaluable=False,
            notes=['Shoulder reference unavailable; contact distance cannot be normalized.'],
        )

    evidence_candidates = geometry.get('contact_evidence', [])
    if not evidence_candidates:
        return build_scored_rule_result(
            score=None,
            evaluable=False,
            notes=['No usable hand, wrist or arm proxy evidence.'],
        )

    best: dict[str, Any] | None = None
    per_side_scores: dict[str, float] = {}
    for evidence in evidence_candidates:
        for landmark_name, point in evidence.get('named_points', []):
            point_score = score_contact_point(rule_name, point, geometry)
            if point_score is None:
                continue
            final_score = clamp_value(point_score['score'] * float(evidence.get('quality', 0.0)))
            per_side_scores[evidence['side']] = max(per_side_scores.get(evidence['side'], 0.0), final_score)
            candidate = {
                'score': final_score,
                'raw_distance_ratio': point_score['raw_distance_ratio'],
                'zone_score': point_score['zone_score'],
                'source': evidence.get('source'),
                'quality': evidence.get('quality'),
                'closest_side': evidence.get('side'),
                'landmarks_used': list(dict.fromkeys([*evidence.get('landmarks_used', []), landmark_name])),
                'notes': evidence.get('notes', []),
                'components': {
                    **point_score['components'],
                    'unweighted_score': point_score['score'],
                    'evidence_quality': evidence.get('quality'),
                    'target_name': point_score.get('target_name'),
                    'per_side_scores': per_side_scores,
                },
            }
            if best is None or candidate['score'] > best['score']:
                best = candidate

    if best is None:
        return build_scored_rule_result(
            score=None,
            evaluable=False,
            notes=['Rule had evidence candidates but no valid region geometry.'],
        )

    return build_scored_rule_result(
        score=best['score'],
        evaluable=True,
        raw_distance_ratio=best['raw_distance_ratio'],
        zone_score=best['zone_score'],
        evidence_quality=best['quality'],
        source=best['source'],
        closest_side=best['closest_side'],
        landmarks_used=best['landmarks_used'],
        score_components=best['components'],
        notes=best['notes'],
    )


def evaluate_head_down(geometry: Mapping[str, Any]) -> dict[str, Any]:
    score = geometry.get('head_tilt_score')
    components = dict(geometry.get('head_tilt_components', {}))
    notes = []
    if components.get('face_pitch_score') is None:
        notes.append('Face pitch unavailable; pose-only head tilt score is capped.')
    return build_scored_rule_result(
        score=score,
        evaluable=score is not None,
        source=components.get('head_tilt_basis'),
        score_components=components,
        notes=notes,
    )


def evaluate_forward_head(geometry: Mapping[str, Any]) -> dict[str, Any]:
    world_ratio = geometry.get('forward_head_world_ratio')
    if world_ratio is None:
        return build_scored_rule_result(
            score=None,
            evaluable=False,
            source='world_pose_unavailable',
            notes=['Pose world landmarks unavailable for forward-head estimate.'],
        )
    score = score_ratio_above(world_ratio, 0.20, 0.60)
    return build_scored_rule_result(
        score=score,
        evaluable=True,
        raw_distance_ratio=world_ratio,
        source='pose_world',
        score_components={'forward_head_world_ratio': world_ratio},
    )


def evaluate_rounded_shoulders_or_asymmetry(geometry: Mapping[str, Any]) -> dict[str, Any]:
    left_ratio = geometry.get('left_shoulder_ear_ratio')
    right_ratio = geometry.get('right_shoulder_ear_ratio')
    asymmetry_ratio = geometry.get('shoulder_asymmetry_ratio')

    ear_ratios = [ratio for ratio in [left_ratio, right_ratio] if ratio is not None]
    shrug_score = None
    if ear_ratios:
        shrug_score = score_ratio_below(float(np.mean(ear_ratios)), 0.30, 0.45)
    asymmetry_score = score_ratio_above(asymmetry_ratio, 0.08, 0.18)
    if shrug_score is None and asymmetry_ratio is None:
        return build_scored_rule_result(
            score=None,
            evaluable=False,
            notes=['Shoulder-ear and shoulder asymmetry cues unavailable.'],
        )

    score = max([value for value in [shrug_score, asymmetry_score] if value is not None], default=0.0)
    return build_scored_rule_result(
        score=score,
        evaluable=True,
        source='pose_2d',
        score_components={
            'shrug_score': shrug_score,
            'asymmetry_score': asymmetry_score,
            'left_shoulder_ear_ratio': left_ratio,
            'right_shoulder_ear_ratio': right_ratio,
            'shoulder_asymmetry_ratio': asymmetry_ratio,
        },
    )


def evaluate_rules(geometry: Mapping[str, Any]) -> dict[str, dict[str, Any]]:
    return {
        'hand_on_face': evaluate_contact_rule('hand_on_face', geometry),
        'hand_on_neck': evaluate_contact_rule('hand_on_neck', geometry),
        'hand_on_chest': evaluate_contact_rule('hand_on_chest', geometry),
        'head_down': evaluate_head_down(geometry),
        'forward_head': evaluate_forward_head(geometry),
        'rounded_shoulders_or_asymmetry': evaluate_rounded_shoulders_or_asymmetry(geometry),
    }


## 9) Scoring por frame, janelas deslizantes e serialização

O frame passa a usar o maior score individual como critério primário. As janelas continuam agregando recorrência temporal, mas o JSON preserva os scores concorrentes de cada regra.


In [9]:
# Consolida scores por frame, agrega janelas e serializa os artefatos JSON em UTF-8.
def pretty_rule_name(rule_name: str | None) -> str | None:
    if rule_name is None:
        return None
    return RULE_DISPLAY_NAMES.get(rule_name, rule_name.replace('_', ' '))


def rule_score(rule: Mapping[str, Any]) -> float:
    return float(rule.get('score') or 0.0)


def rule_is_active(rule: Mapping[str, Any]) -> bool:
    return bool(rule.get('passed_threshold'))


def choose_primary_trigger(rules: Mapping[str, Mapping[str, Any]]) -> str | None:
    known_rules = [
        name for name, rule in rules.items()
        if rule.get('state') != 'unknown'
    ]
    if not known_rules:
        return None
    return max(
        known_rules,
        key=lambda name: (rule_score(rules[name]), -RULE_ORDER.index(name) if name in RULE_ORDER else -999),
    )


def display_rules_for_overlay(rules: Mapping[str, Mapping[str, Any]]) -> list[dict[str, Any]]:
    visible_rules = [
        {
            'name': name,
            'label': pretty_rule_name(name),
            'score': optional_round(rule_score(rule), 3),
            'source': rule.get('source'),
        }
        for name, rule in rules.items()
        if rule_score(rule) >= DISPLAY_SCORE_THRESHOLD and rule.get('state') != 'unknown'
    ]
    return sorted(visible_rules, key=lambda item: item['score'] or 0.0, reverse=True)


def build_frame_explanation(rules: Mapping[str, Mapping[str, Any]], frame_level: str) -> str:
    if frame_level == 'insufficient_data':
        return 'insufficient data'
    display_rules = display_rules_for_overlay(rules)
    if not display_rules:
        return 'no major posture signal'
    top_rule = display_rules[0]
    return f"{top_rule['label']} score {top_rule['score']:.2f}"


def evidence_summary(evidence: Mapping[str, Any]) -> dict[str, Any]:
    return {
        'source': evidence.get('source'),
        'quality': optional_round(evidence.get('quality')),
        'landmark_count': len(evidence.get('named_points', [])),
        'landmarks_used': evidence.get('landmarks_used', []),
    }


def build_frame_signal_record(
    frame_idx: int,
    timestamp_s: float,
    geometry: Mapping[str, Any],
    rules: Mapping[str, Mapping[str, Any]],
) -> dict[str, Any]:
    known_rules = [name for name, rule in rules.items() if rule.get('state') != 'unknown']
    unknown_rules = [name for name, rule in rules.items() if rule.get('state') == 'unknown']
    active_rules = [name for name, rule in rules.items() if rule_is_active(rule)]
    display_rules = display_rules_for_overlay(rules)
    frame_score = max([rule_score(rule) for rule in rules.values() if rule.get('state') != 'unknown'], default=0.0)

    partial_data = bool(known_rules) and bool(unknown_rules)
    insufficient_data = (not geometry.get('shoulder_reference_ok')) or not known_rules
    if insufficient_data:
        frame_level = 'insufficient_data'
    elif frame_score >= FRAME_STRONG_SCORE_THRESHOLD:
        frame_level = 'strong_signal'
    elif frame_score >= RULE_SCORE_THRESHOLD:
        frame_level = 'possible_signal'
    else:
        frame_level = 'ok'

    primary_trigger = choose_primary_trigger(rules)
    explanation = build_frame_explanation(rules, frame_level)
    head_components = geometry.get('head_tilt_components', {})
    shoulder_mid = geometry.get('shoulder_mid')

    metrics = {
        'shoulder_width': optional_round(geometry.get('shoulder_width')),
        'shoulder_mid_x': optional_round(shoulder_mid.x if shoulder_mid is not None else None),
        'shoulder_mid_y': optional_round(shoulder_mid.y if shoulder_mid is not None else None),
        'shoulder_to_nose': optional_round(geometry.get('shoulder_to_nose')),
        'head_height_ratio': optional_round(geometry.get('head_height_ratio')),
        'head_tilt_score': optional_round(geometry.get('head_tilt_score')),
        'face_pitch_score': optional_round(head_components.get('face_pitch_score')),
        'face_pitch_degrees': optional_round(head_components.get('face_pitch_degrees')),
        'head_drop_score': optional_round(head_components.get('head_drop_score')),
        'shoulder_ear_compression_score': optional_round(head_components.get('shoulder_ear_compression_score')),
        'neck_fraction': optional_round(geometry.get('neck_fraction')),
        'neck_top_y': optional_round(geometry.get('neck_top_y')),
        'neck_bottom_y': optional_round(geometry.get('neck_bottom_y')),
        'forward_head_world_ratio': optional_round(geometry.get('forward_head_world_ratio')),
        'left_shoulder_ear_ratio': optional_round(geometry.get('left_shoulder_ear_ratio')),
        'right_shoulder_ear_ratio': optional_round(geometry.get('right_shoulder_ear_ratio')),
        'shoulder_asymmetry_ratio': optional_round(geometry.get('shoulder_asymmetry_ratio')),
        'face_anchor_count': int(geometry.get('face_anchor_count', 0)),
        'left_hand_visible': bool(geometry.get('left_hand_visible')),
        'right_hand_visible': bool(geometry.get('right_hand_visible')),
        'left_wrist_visible': bool(geometry.get('left_wrist_visible')),
        'right_wrist_visible': bool(geometry.get('right_wrist_visible')),
        'left_elbow_visible': bool(geometry.get('left_elbow_visible')),
        'right_elbow_visible': bool(geometry.get('right_elbow_visible')),
        'left_evidence': evidence_summary(geometry.get('left_evidence', {})),
        'right_evidence': evidence_summary(geometry.get('right_evidence', {})),
        'coordinate_mode': geometry.get('coordinate_mode'),
    }

    return {
        'frame_idx': int(frame_idx),
        'timestamp_s': round(float(timestamp_s), 3),
        'status': 'insufficient_data' if insufficient_data else 'scorable',
        'partial_data': partial_data,
        'metrics': metrics,
        'rules': dict(rules),
        'frame_score': round(float(frame_score), 3),
        'frame_score_basis': 'max_individual_rule_score',
        'frame_level': frame_level,
        'explanation': explanation,
        'primary_trigger': primary_trigger,
        'active_rules': active_rules,
        'display_rules': display_rules,
    }


def window_ranges(duration_seconds: float, window_seconds: float, stride_seconds: float) -> list[tuple[int, float, float]]:
    if duration_seconds <= 0:
        return []
    if duration_seconds <= window_seconds:
        return [(0, 0.0, duration_seconds)]

    ranges: list[tuple[int, float, float]] = []
    window_idx = 0
    start_s = 0.0
    while start_s < duration_seconds:
        end_s = min(duration_seconds, start_s + window_seconds)
        ranges.append((window_idx, start_s, end_s))
        if end_s >= duration_seconds:
            break
        window_idx += 1
        start_s += stride_seconds
    return ranges


def count_max_consecutive_signal_frames(frame_records: Iterable[Mapping[str, Any]]) -> int:
    best_run = 0
    current_run = 0
    for frame in frame_records:
        if frame.get('frame_level') in {'possible_signal', 'strong_signal'}:
            current_run += 1
            best_run = max(best_run, current_run)
        else:
            current_run = 0
    return best_run


def label_window(signal_ratio: float | None) -> str:
    if signal_ratio is None:
        return 'insufficient_data'
    if signal_ratio > 0.35:
        return 'strong_signal'
    if signal_ratio >= 0.15:
        return 'possible_signal'
    return 'ok'


def aggregate_windows(frame_records: list[dict[str, Any]], duration_seconds: float) -> list[dict[str, Any]]:
    summaries: list[dict[str, Any]] = []
    for window_idx, start_s, end_s in window_ranges(duration_seconds, WINDOW_SECONDS, WINDOW_STRIDE_SECONDS):
        window_frames = [
            frame
            for frame in frame_records
            if start_s <= frame.get('timestamp_s', 0.0) < (end_s + 1e-9)
        ]

        scorable_frames = [frame for frame in window_frames if frame.get('frame_level') != 'insufficient_data']
        frames_total = len(window_frames)
        frames_scorable = len(scorable_frames)
        frames_insufficient = frames_total - frames_scorable

        signal_frames = [
            frame for frame in scorable_frames if frame.get('frame_level') in {'possible_signal', 'strong_signal'}
        ]
        strong_frames = [frame for frame in scorable_frames if frame.get('frame_level') == 'strong_signal']
        signal_ratio = (len(signal_frames) / frames_scorable) if frames_scorable else None
        strong_signal_ratio = (len(strong_frames) / frames_scorable) if frames_scorable else None
        frame_scores = [float(frame.get('frame_score', 0.0)) for frame in scorable_frames]
        trigger_counter = Counter(
            frame.get('primary_trigger') for frame in signal_frames if frame.get('primary_trigger') is not None
        )
        most_common_trigger = trigger_counter.most_common(1)[0][0] if trigger_counter else None
        window_level = label_window(signal_ratio)

        if window_level == 'insufficient_data':
            explanation = 'insufficient data coverage'
        elif most_common_trigger is not None:
            explanation = f"{pretty_rule_name(most_common_trigger)} recurring across the window"
        else:
            explanation = 'no major posture signal'

        summaries.append({
            'window_idx': window_idx,
            'start_s': round(start_s, 3),
            'end_s': round(end_s, 3),
            'frames_total': frames_total,
            'frames_scorable': frames_scorable,
            'frames_insufficient': frames_insufficient,
            'signal_ratio': optional_round(signal_ratio),
            'strong_signal_ratio': optional_round(strong_signal_ratio),
            'max_consecutive_signal_frames': count_max_consecutive_signal_frames(window_frames),
            'most_common_trigger': most_common_trigger,
            'window_score_mean': optional_round(float(np.mean(frame_scores)) if frame_scores else None),
            'window_score_peak': optional_round(float(np.max(frame_scores)) if frame_scores else None),
            'window_score_basis': 'frame max_individual_rule_score',
            'window_level': window_level,
            'explanation': explanation,
        })
    return summaries


def peak_window_level(window_summaries: Iterable[Mapping[str, Any]]) -> str:
    best_level = 'insufficient_data'
    best_score = WINDOW_LEVEL_SEVERITY[best_level]
    for summary in window_summaries:
        level = summary.get('window_level', 'insufficient_data')
        if WINDOW_LEVEL_SEVERITY.get(level, -1) > best_score:
            best_level = level
            best_score = WINDOW_LEVEL_SEVERITY[level]
    return best_level


def has_adjacent_strong_windows(valid_windows: list[Mapping[str, Any]]) -> bool:
    run_length = 0
    previous_index = None
    for window in valid_windows:
        if window.get('window_level') == 'strong_signal':
            if previous_index is not None and window.get('window_idx') == previous_index + 1:
                run_length += 1
            else:
                run_length = 1
            if run_length >= 2:
                return True
        else:
            run_length = 0
        previous_index = window.get('window_idx')
    return False


def build_video_summary(frame_records: list[dict[str, Any]], window_summaries: list[dict[str, Any]]) -> dict[str, Any]:
    sampled_frames = len(frame_records)
    scorable_frames = sum(frame.get('status') == 'scorable' for frame in frame_records)
    coverage_ratio = (scorable_frames / sampled_frames) if sampled_frames else 0.0

    valid_windows = [window for window in window_summaries if window.get('window_level') != 'insufficient_data']
    trigger_counter = Counter(
        window.get('most_common_trigger') for window in valid_windows if window.get('most_common_trigger') is not None
    )
    most_common_trigger = trigger_counter.most_common(1)[0][0] if trigger_counter else None

    if not valid_windows:
        return {
            'level': 'insufficient_data',
            'video_signal_ratio': None,
            'video_strong_window_ratio': None,
            'coverage_ratio': optional_round(coverage_ratio),
            'peak_window_level': 'insufficient_data',
            'most_common_trigger': most_common_trigger,
            'dominant_explanation': 'insufficient upper-body coverage for reliable screening',
            'explanation': 'insufficient upper-body coverage for reliable screening',
        }

    strong_windows = [window for window in valid_windows if window.get('window_level') == 'strong_signal']
    signal_windows = [
        window for window in valid_windows if window.get('window_level') in {'possible_signal', 'strong_signal'}
    ]

    video_signal_ratio = len(signal_windows) / len(valid_windows)
    video_strong_window_ratio = len(strong_windows) / len(valid_windows)
    if (video_strong_window_ratio > 0.35) or has_adjacent_strong_windows(valid_windows):
        level = 'strong_signal'
    elif video_signal_ratio >= 0.15:
        level = 'possible_signal'
    else:
        level = 'ok'

    if coverage_ratio < 0.10:
        level = 'insufficient_data'
        dominant_explanation = 'insufficient upper-body coverage for reliable screening'
    elif most_common_trigger is not None:
        dominant_explanation = f"{pretty_rule_name(most_common_trigger)} is the most frequent trigger"
    else:
        dominant_explanation = 'no major posture signal'

    return {
        'level': level,
        'video_signal_ratio': optional_round(video_signal_ratio),
        'video_strong_window_ratio': optional_round(video_strong_window_ratio),
        'coverage_ratio': optional_round(coverage_ratio),
        'peak_window_level': peak_window_level(valid_windows),
        'most_common_trigger': most_common_trigger,
        'dominant_explanation': dominant_explanation,
        'explanation': dominant_explanation,
    }


def percentage(count: int, total: int) -> float:
    if total <= 0:
        return 0.0
    return round((count / total) * 100.0, 1)


def build_limitations(frame_records: list[dict[str, Any]]) -> list[str]:
    limitations = [
        'This is a heuristic geometric posture screener based on MediaPipe Holistic landmarks.',
        'The v2 pipeline does not reason about multiple visible people.',
        'Scores are heuristic 0-1 signals, not calibrated statistical probabilities.',
    ]

    total_frames = len(frame_records)
    if total_frames == 0:
        limitations.append('No sampled frames were processed from the source video.')
        return limitations

    insufficient_frames = sum(frame.get('status') == 'insufficient_data' for frame in frame_records)
    partial_frames = sum(bool(frame.get('partial_data')) for frame in frame_records)
    no_direct_hand_frames = sum(
        not frame.get('metrics', {}).get('left_hand_visible') and not frame.get('metrics', {}).get('right_hand_visible')
        for frame in frame_records
    )
    fallback_frames = sum(
        any(
            frame.get('metrics', {}).get(side, {}).get('source') in {'pose_wrist_fallback', 'arm_proxy'}
            for side in ['left_evidence', 'right_evidence']
        )
        for frame in frame_records
    )
    face_pitch_missing_frames = sum(
        frame.get('metrics', {}).get('face_pitch_score') is None
        for frame in frame_records
    )

    if insufficient_frames:
        limitations.append(
            f"Shoulder reference was insufficient in {percentage(insufficient_frames, total_frames)}% of sampled frames."
        )
    if partial_frames:
        limitations.append(
            f"At least one rule was unknown in {percentage(partial_frames, total_frames)}% of sampled frames due to partial landmark coverage."
        )
    if no_direct_hand_frames:
        limitations.append(
            f"Holistic hand landmarks were unavailable in {percentage(no_direct_hand_frames, total_frames)}% of sampled frames."
        )
    if fallback_frames:
        limitations.append(
            f"Contact rules used pose wrist/elbow fallback in {percentage(fallback_frames, total_frames)}% of sampled frames."
        )
    if face_pitch_missing_frames:
        limitations.append(
            f"Head tilt score lacked face-pitch evidence in {percentage(face_pitch_missing_frames, total_frames)}% of sampled frames."
        )
    return limitations


def sanitize_for_json(value: Any) -> Any:
    if isinstance(value, dict):
        return {key: sanitize_for_json(item) for key, item in value.items()}
    if isinstance(value, list):
        return [sanitize_for_json(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    return value


def write_payload_json(payload: Mapping[str, Any], json_output_path: Path) -> None:
    json_output_path.parent.mkdir(parents=True, exist_ok=True)
    json_output_path.write_text(
        json.dumps(sanitize_for_json(dict(payload)), ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )


## 10) Renderização do vídeo anotado

O overlay v2 remove o triângulo nariz-ombros, desenha conexões anatômicas de braços e mãos completas, e mostra somente regras cujo score passou o limiar único configurado.


In [10]:
# Desenha landmarks anatômicos, faixa dinâmica de pescoço e painel com regras acima do limiar único.
POSE_DRAW_SEGMENTS = [
    ('left_shoulder', 'right_shoulder'),
    ('left_shoulder', 'left_elbow'),
    ('left_elbow', 'left_wrist'),
    ('right_shoulder', 'right_elbow'),
    ('right_elbow', 'right_wrist'),
    ('left_shoulder', 'left_hip'),
    ('right_shoulder', 'right_hip'),
    ('left_hip', 'right_hip'),
    ('left_eye_outer', 'left_ear'),
    ('right_eye_outer', 'right_ear'),
    ('mouth_left', 'mouth_right'),
]

HAND_DRAW_SEGMENTS = [
    ('wrist', 'thumb_cmc'),
    ('thumb_cmc', 'thumb_mcp'),
    ('thumb_mcp', 'thumb_ip'),
    ('thumb_ip', 'thumb_tip'),
    ('wrist', 'index_mcp'),
    ('index_mcp', 'index_pip'),
    ('index_pip', 'index_dip'),
    ('index_dip', 'index_tip'),
    ('wrist', 'middle_mcp'),
    ('middle_mcp', 'middle_pip'),
    ('middle_pip', 'middle_dip'),
    ('middle_dip', 'middle_tip'),
    ('wrist', 'ring_mcp'),
    ('ring_mcp', 'ring_pip'),
    ('ring_pip', 'ring_dip'),
    ('ring_dip', 'ring_tip'),
    ('wrist', 'pinky_mcp'),
    ('pinky_mcp', 'pinky_pip'),
    ('pinky_pip', 'pinky_dip'),
    ('pinky_dip', 'pinky_tip'),
]

FACE_DRAW_SEGMENTS = [
    ('face_left_eye_outer', 'face_left_eye_inner'),
    ('face_right_eye_inner', 'face_right_eye_outer'),
    ('face_mouth_left', 'face_mouth_right'),
    ('face_jaw_left', 'face_chin'),
    ('face_chin', 'face_jaw_right'),
]


def point_to_pixel(point: PointData | None, frame_shape: tuple[int, int, int]) -> tuple[int, int] | None:
    if point is None:
        return None
    height, width = frame_shape[:2]
    x = int(np.clip(point.x, 0.0, 1.0) * width)
    y = int(np.clip(point.y, 0.0, 1.0) * height)
    return x, y


def normalized_to_pixel(x: float | None, y: float | None, frame_shape: tuple[int, int, int]) -> tuple[int, int] | None:
    if x is None or y is None:
        return None
    height, width = frame_shape[:2]
    return int(np.clip(x, 0.0, 1.0) * width), int(np.clip(y, 0.0, 1.0) * height)


def draw_segment(
    frame: np.ndarray,
    point_a: PointData | None,
    point_b: PointData | None,
    color: tuple[int, int, int],
    thickness: int = 2,
) -> None:
    pixel_a = point_to_pixel(point_a, frame.shape)
    pixel_b = point_to_pixel(point_b, frame.shape)
    if pixel_a is None or pixel_b is None:
        return
    cv2.line(frame, pixel_a, pixel_b, color, thickness, cv2.LINE_AA)


def draw_points(frame: np.ndarray, points: Mapping[str, PointData], color: tuple[int, int, int], radius: int = 4) -> None:
    for point in points.values():
        pixel = point_to_pixel(point, frame.shape)
        if pixel is None:
            continue
        cv2.circle(frame, pixel, radius, color, -1, cv2.LINE_AA)


def draw_landmarks_overlay(frame: np.ndarray, drawing_points: Mapping[str, dict[str, PointData]]) -> None:
    pose_points = drawing_points.get('pose_2d', {})
    face_points = drawing_points.get('face_2d', {})
    left_hand = drawing_points.get('left_hand_2d', {})
    right_hand = drawing_points.get('right_hand_2d', {})

    for start_name, end_name in POSE_DRAW_SEGMENTS:
        draw_segment(frame, pose_points.get(start_name), pose_points.get(end_name), (255, 210, 0), 2)
    for start_name, end_name in FACE_DRAW_SEGMENTS:
        draw_segment(frame, face_points.get(start_name), face_points.get(end_name), (210, 170, 255), 1)
    for start_name, end_name in HAND_DRAW_SEGMENTS:
        draw_segment(frame, left_hand.get(start_name), left_hand.get(end_name), (80, 220, 80), 2)
        draw_segment(frame, right_hand.get(start_name), right_hand.get(end_name), (0, 170, 255), 2)

    draw_points(frame, pose_points, (255, 210, 0), radius=4)
    draw_points(frame, face_points, (210, 170, 255), radius=2)
    draw_points(frame, left_hand, (80, 220, 80), radius=3)
    draw_points(frame, right_hand, (0, 170, 255), radius=3)


def draw_neck_debug_region(frame: np.ndarray, frame_record: Mapping[str, Any] | None) -> None:
    if frame_record is None:
        return
    metrics = frame_record.get('metrics', {})
    shoulder_mid_x = metrics.get('shoulder_mid_x')
    shoulder_width = metrics.get('shoulder_width')
    neck_top_y = metrics.get('neck_top_y')
    neck_bottom_y = metrics.get('neck_bottom_y')
    if None in {shoulder_mid_x, shoulder_width, neck_top_y, neck_bottom_y}:
        return

    left_x = shoulder_mid_x - (shoulder_width * 0.60)
    right_x = shoulder_mid_x + (shoulder_width * 0.60)
    top_left = normalized_to_pixel(left_x, neck_top_y, frame.shape)
    top_right = normalized_to_pixel(right_x, neck_top_y, frame.shape)
    bottom_left = normalized_to_pixel(left_x, neck_bottom_y, frame.shape)
    bottom_right = normalized_to_pixel(right_x, neck_bottom_y, frame.shape)
    if None in {top_left, top_right, bottom_left, bottom_right}:
        return

    overlay = frame.copy()
    polygon = np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.int32)
    cv2.fillPoly(overlay, [polygon], (255, 0, 255))
    cv2.addWeighted(overlay, 0.12, frame, 0.88, 0.0, frame)
    cv2.polylines(frame, [polygon], isClosed=True, color=(255, 0, 255), thickness=1, lineType=cv2.LINE_AA)


def select_window_for_timestamp(timestamp_s: float, window_summaries: Iterable[Mapping[str, Any]]) -> dict[str, Any] | None:
    active_windows = [
        window
        for window in window_summaries
        if window.get('start_s', 0.0) <= timestamp_s < (window.get('end_s', 0.0) + 1e-9)
    ]
    if not active_windows:
        return None
    return max(
        active_windows,
        key=lambda window: (
            WINDOW_LEVEL_SEVERITY.get(window.get('window_level', 'insufficient_data'), -1),
            window.get('start_s', 0.0),
        ),
    )


def format_display_rules(frame_record: Mapping[str, Any]) -> str:
    display_rules = frame_record.get('display_rules', [])
    if not display_rules:
        return 'none'
    chunks = []
    for item in display_rules[:4]:
        label = item.get('label') or item.get('name')
        score = float(item.get('score') or 0.0)
        chunks.append(f'{label} {score:.2f}')
    return ', '.join(chunks)


def draw_status_panel(
    frame: np.ndarray,
    frame_record: Mapping[str, Any] | None,
    window_record: Mapping[str, Any] | None,
    *,
    exact_sample: bool,
) -> None:
    if frame_record is None:
        frame_record = {
            'frame_level': 'insufficient_data',
            'frame_score': 0.0,
            'display_rules': [],
            'explanation': 'insufficient data',
        }

    frame_level = frame_record.get('frame_level', 'insufficient_data')
    panel_color = FRAME_LEVEL_COLORS.get(frame_level, (120, 120, 120))
    window_level = window_record.get('window_level', 'n/a') if window_record is not None else 'n/a'
    window_explanation = window_record.get('explanation', 'n/a') if window_record is not None else 'n/a'

    lines = [
        f"Frame level: {frame_level}",
        f"Max rule score: {float(frame_record.get('frame_score', 0.0)):.2f}",
        f"Rules >= {DISPLAY_SCORE_THRESHOLD:.2f}: {format_display_rules(frame_record)}",
        f"Window level: {window_level}",
        f"Explanation: {frame_record.get('explanation', 'n/a')}",
    ]
    if frame_record.get('status') == 'insufficient_data':
        lines[0] = 'Frame level: insufficient data'
    if not exact_sample:
        lines.append('Overlay source: last sampled state')
    if window_record is not None:
        lines.append(f"Window note: {window_explanation}")

    overlay = frame.copy()
    panel_x, panel_y = 20, 20
    panel_width = min(960, frame.shape[1] - 40)
    panel_height = 34 + (30 * len(lines))
    cv2.rectangle(overlay, (panel_x, panel_y), (panel_x + panel_width, panel_y + panel_height), (10, 10, 10), -1)
    cv2.addWeighted(overlay, 0.60, frame, 0.40, 0.0, frame)
    cv2.rectangle(frame, (panel_x, panel_y), (panel_x + panel_width, panel_y + panel_height), panel_color, 2)

    for line_idx, text in enumerate(lines, start=1):
        cv2.putText(
            frame,
            text[:120],
            (panel_x + 12, panel_y + (line_idx * 26)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.64,
            (245, 245, 245),
            2,
            cv2.LINE_AA,
        )


def render_annotated_video(
    source_path: Path,
    annotated_video_path: Path,
    frame_records: list[dict[str, Any]],
    drawings_by_frame: Mapping[int, dict[str, dict[str, PointData]]],
    window_summaries: list[dict[str, Any]],
    source_fps: float,
    frame_size: tuple[int, int],
) -> None:
    capture = cv2.VideoCapture(str(source_path))
    if not capture.isOpened():
        raise RuntimeError(f'Could not reopen source video for annotation: {source_path}')

    writer = cv2.VideoWriter(
        str(annotated_video_path),
        cv2.VideoWriter_fourcc(*'mp4v'),
        source_fps,
        frame_size,
    )
    if not writer.isOpened():
        capture.release()
        raise RuntimeError(f'Could not create annotated video: {annotated_video_path}')

    sorted_frames = sorted(frame_records, key=lambda frame: frame['frame_idx'])
    current_record: dict[str, Any] | None = None
    sample_cursor = 0
    frame_idx = 0

    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break

            while sample_cursor < len(sorted_frames) and sorted_frames[sample_cursor]['frame_idx'] <= frame_idx:
                current_record = sorted_frames[sample_cursor]
                sample_cursor += 1

            exact_sample = current_record is not None and current_record['frame_idx'] == frame_idx
            if current_record is not None:
                drawing_points = drawings_by_frame.get(current_record['frame_idx'])
                if drawing_points is not None:
                    draw_landmarks_overlay(frame, drawing_points)
                draw_neck_debug_region(frame, current_record)
                window_record = select_window_for_timestamp(frame_idx / source_fps, window_summaries)
                draw_status_panel(frame, current_record, window_record, exact_sample=exact_sample)
            else:
                draw_status_panel(frame, None, None, exact_sample=False)

            writer.write(frame)
            frame_idx += 1
    finally:
        capture.release()
        writer.release()


## 11) Pipeline principal `process_video_asset(...)`

Esta célula executa o fluxo de ponta a ponta com Holistic Tasks: leitura do vídeo, amostragem, inferência, suavização, scoring, agregação, JSON e MP4 anotado.


In [13]:
# Executa o pipeline v2 com um único HolisticLandmarker em modo VIDEO.
from time import time


def ensure_runtime_ready() -> None:
    if mp is None:
        raise ImportError('MediaPipe is not available. Execute the installation cell and rerun the imports.')
    if not hasattr(mp, 'tasks') or not hasattr(mp.tasks.vision, 'HolisticLandmarker'):
        raise ImportError('The installed mediapipe package does not expose HolisticLandmarker from the Tasks API.')


def resolve_video_input_path(video_path: str | Path) -> Path:
    candidate = Path(video_path)
    if candidate.is_absolute():
        return candidate.resolve()

    search_roots = [Path.cwd(), REPO_ROOT, NOTEBOOK_ROOT]
    for root in search_roots:
        resolved = (root / candidate).resolve()
        if resolved.exists():
            return resolved
    return (NOTEBOOK_ROOT / candidate).resolve()


def create_holistic_landmarker() -> Any:
    base_options = mp.tasks.BaseOptions(model_asset_path=str(HOLISTIC_LANDMARKER_MODEL_PATH))
    options = mp.tasks.vision.HolisticLandmarkerOptions(
        base_options=base_options,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        min_face_detection_confidence=0.5,
        min_face_suppression_threshold=0.5,
        min_face_landmarks_confidence=0.5,
        min_pose_detection_confidence=0.5,
        min_pose_suppression_threshold=0.5,
        min_pose_landmarks_confidence=0.5,
        min_hand_landmarks_confidence=0.5,
        output_face_blendshapes=False,
        output_segmentation_mask=False,
    )
    return mp.tasks.vision.HolisticLandmarker.create_from_options(options)


def frame_to_mp_image(frame_rgb: np.ndarray) -> Any:
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)


def build_pipeline_metadata(coordinate_mode: str) -> dict[str, Any]:
    return {
        'backend': PIPELINE_BACKEND,
        'model_asset_path': str(HOLISTIC_LANDMARKER_MODEL_PATH),
        'model_asset_url': HOLISTIC_LANDMARKER_MODEL_URL,
        'coordinate_mode': coordinate_mode,
        'visibility_threshold': VISIBILITY_THRESHOLD,
        'ema_alpha': EMA_ALPHA,
        'window_seconds': WINDOW_SECONDS,
        'window_stride_seconds': WINDOW_STRIDE_SECONDS,
        'rule_score_threshold': RULE_SCORE_THRESHOLD,
        'display_score_threshold': DISPLAY_SCORE_THRESHOLD,
        'frame_score_basis': 'max_individual_rule_score',
        'neck_parameters': {
            'base_neck_fraction': BASE_NECK_FRACTION,
            'max_neck_tilt_bonus': MAX_NECK_TILT_BONUS,
            'max_neck_fraction': MAX_NECK_FRACTION,
            'chest_overlap_fraction': CHEST_OVERLAP_FRACTION,
            'head_drop_neutral_ratio': HEAD_DROP_NEUTRAL_RATIO,
            'head_drop_strong_ratio': HEAD_DROP_STRONG_RATIO,
        },
        'hand_source_quality': HAND_SOURCE_QUALITY,
    }


def build_fail_safe_payload(
    source_path: Path,
    json_output_path: Path,
    annotated_video_path: Path | None,
    *,
    error_message: str,
    source_fps: float = 0.0,
    duration_seconds: float = 0.0,
    frames_total: int = 0,
    frames_sampled: int = 0,
) -> dict[str, Any]:
    payload = {
        'schema_version': SCHEMA_VERSION,
        'video': {
            'video_id': source_path.stem or 'video',
            'source_path': str(source_path),
            'source_name': source_path.name,
            'annotated_video_path': str(annotated_video_path) if annotated_video_path is not None else None,
            'source_fps': optional_round(source_fps),
            'sample_fps': optional_round(TARGET_SAMPLE_FPS),
            'duration_seconds': optional_round(duration_seconds),
            'frames_total': int(frames_total),
            'frames_sampled': int(frames_sampled),
        },
        'pipeline': build_pipeline_metadata(COORDINATE_MODE_2D),
        'limitations': [
            'This is a heuristic geometric posture screener based on MediaPipe Holistic landmarks.',
            'The output reports observable posture and self-touch patterns only; it does not infer emotion, deception, diagnosis, or intent.',
            error_message,
        ],
        'frame_signals': [],
        'window_summaries': [],
        'video_summary': {
            'level': 'insufficient_data',
            'video_signal_ratio': None,
            'video_strong_window_ratio': None,
            'coverage_ratio': 0.0,
            'peak_window_level': 'insufficient_data',
            'most_common_trigger': None,
            'dominant_explanation': 'insufficient upper-body coverage for reliable screening',
            'explanation': 'insufficient upper-body coverage for reliable screening',
        },
    }
    write_payload_json(payload, json_output_path)
    return payload


def process_video_asset(video_path: str | Path) -> dict[str, Any]:
    ensure_runtime_ready()
    assert_model_assets_available()

    source_path = resolve_video_input_path(video_path)
    video_id = source_path.stem or 'video'
    timestamp_ms = int(round(time() * 1000.0))
    json_output_path = OUTPUT_JSON_DIR / f'{video_id}.{timestamp_ms}.signals.json'
    annotated_video_path = OUTPUT_VIDEO_DIR / f'{video_id}.{timestamp_ms}.annotated.mp4'

    if not source_path.exists():
        return build_fail_safe_payload(
            source_path,
            json_output_path,
            annotated_video_path,
            error_message='Source video path does not exist.',
        )

    capture = cv2.VideoCapture(str(source_path))
    if not capture.isOpened():
        return build_fail_safe_payload(
            source_path,
            json_output_path,
            annotated_video_path,
            error_message='OpenCV could not open the source video.',
        )

    source_fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0) or 30.0
    frames_total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration_seconds = (frames_total / source_fps) if source_fps > 0 else 0.0
    frame_step = max(1, round(source_fps / TARGET_SAMPLE_FPS))
    effective_sample_fps = source_fps / frame_step

    frame_records: list[dict[str, Any]] = []
    drawings_by_frame: dict[int, dict[str, dict[str, PointData]]] = {}
    coordinate_modes_used: Counter[str] = Counter()
    warnings: list[str] = []

    smoothers = build_smoothers()
    frame_idx = 0
    progress_bar = tqdm(total=frames_total or None, desc=f'Processing v2 {video_id}', unit='frame')

    try:
        with create_holistic_landmarker() as holistic_landmarker:
            while True:
                ok, frame = capture.read()
                if not ok:
                    break
                progress_bar.update(1)

                if frame_idx % frame_step != 0:
                    frame_idx += 1
                    continue

                timestamp_s = frame_idx / source_fps
                timestamp_ms = int(round(timestamp_s * 1000.0))
                try:
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = frame_to_mp_image(frame_rgb)
                    holistic_result = holistic_landmarker.detect_for_video(mp_image, timestamp_ms)
                    raw_sets = extract_landmark_sets(holistic_result)
                except Exception as exc:
                    warnings.append(f'Holistic inference failed at sampled frame {frame_idx}: {exc}')
                    raw_sets = build_empty_landmark_sets()

                smoothed_sets = smooth_landmark_sets(frame_idx, raw_sets, smoothers)
                geometry = compute_reference_geometry(smoothed_sets)
                rules = evaluate_rules(geometry)
                frame_record = build_frame_signal_record(frame_idx, timestamp_s, geometry, rules)
                frame_records.append(frame_record)
                drawings_by_frame[frame_idx] = clone_drawing_points(smoothed_sets)
                coordinate_modes_used[geometry.get('coordinate_mode', COORDINATE_MODE_2D)] += 1

                frame_idx += 1
    finally:
        progress_bar.close()
        capture.release()

    if not frame_records:
        return build_fail_safe_payload(
            source_path,
            json_output_path,
            annotated_video_path,
            error_message='No sampled frames were produced from the source video.',
            source_fps=source_fps,
            duration_seconds=duration_seconds,
            frames_total=frames_total,
            frames_sampled=0,
        )

    window_summaries = aggregate_windows(frame_records, duration_seconds)
    video_summary = build_video_summary(frame_records, window_summaries)
    limitations = build_limitations(frame_records)
    limitations.extend(warnings)

    overall_coordinate_mode = coordinate_modes_used.most_common(1)[0][0] if coordinate_modes_used else COORDINATE_MODE_2D

    try:
        render_annotated_video(
            source_path=source_path,
            annotated_video_path=annotated_video_path,
            frame_records=frame_records,
            drawings_by_frame=drawings_by_frame,
            window_summaries=window_summaries,
            source_fps=source_fps,
            frame_size=(frame_width, frame_height),
        )
        annotated_video_string = str(annotated_video_path)
    except Exception as exc:
        limitations.append(f'Annotated video rendering failed: {exc}')
        annotated_video_string = None

    payload = {
        'schema_version': SCHEMA_VERSION,
        'video': {
            'video_id': video_id,
            'source_path': str(source_path),
            'source_name': source_path.name,
            'annotated_video_path': annotated_video_string,
            'source_fps': optional_round(source_fps),
            'sample_fps': optional_round(effective_sample_fps),
            'duration_seconds': optional_round(duration_seconds),
            'frames_total': int(frames_total),
            'frames_sampled': int(len(frame_records)),
        },
        'pipeline': build_pipeline_metadata(overall_coordinate_mode),
        'limitations': limitations,
        'frame_signals': frame_records,
        'window_summaries': window_summaries,
        'video_summary': video_summary,
    }

    write_payload_json(payload, json_output_path)
    return payload


## 12) Execução em lote sobre `concepts_video/data/video`

A próxima célula varre a pasta de entrada e processa cada vídeo compatível. Execute-a quando quiser gerar os artefatos v2.


In [14]:
# Processa todos os vídeos padrão e monta uma tabela curta com os principais artefatos v2.
VIDEO_PATHS = DEFAULT_VIDEO_PATHS

RUN_RESULTS: list[dict[str, Any]] = []
for index, video_path in enumerate(VIDEO_PATHS, start=1):
    display(Markdown(f'### Processing v2 {index}/{len(VIDEO_PATHS)}: `{video_path.name}`'))
    payload = process_video_asset(video_path)
    RUN_RESULTS.append(payload)

SUMMARY_ROWS = [
    {
        'video_id': payload['video']['video_id'],
        'level': payload['video_summary']['level'],
        'coverage_ratio': payload['video_summary']['coverage_ratio'],
        'video_signal_ratio': payload['video_summary']['video_signal_ratio'],
        'strong_window_ratio': payload['video_summary']['video_strong_window_ratio'],
        'most_common_trigger': payload['video_summary']['most_common_trigger'],
        'json_path': str(OUTPUT_JSON_DIR / f"{payload['video']['video_id']}.signals.json"),
        'annotated_video_path': payload['video']['annotated_video_path'],
    }
    for payload in RUN_RESULTS
]

if pd is not None:
    SUMMARY_TABLE = pd.DataFrame(SUMMARY_ROWS)
else:
    SUMMARY_TABLE = SUMMARY_ROWS
display(SUMMARY_TABLE)


### Processing v2 1/4: `domestic_abuse1.mp4`

Processing v2 domestic_abuse1:   0%|          | 0/384 [00:00<?, ?frame/s]

### Processing v2 2/4: `domestic_abuse2.mp4`

Processing v2 domestic_abuse2:   0%|          | 0/431 [00:00<?, ?frame/s]

### Processing v2 3/4: `sad_woman1.mp4`

Processing v2 sad_woman1:   0%|          | 0/877 [00:00<?, ?frame/s]

### Processing v2 4/4: `sad_woman2.mp4`

Processing v2 sad_woman2:   0%|          | 0/917 [00:00<?, ?frame/s]

,video_id,level,coverage_ratio,video_signal_ratio,strong_window_ratio,most_common_trigger,json_path,annotated_video_path
0,domestic_abuse1,strong_signal,1.0000,1.0,1.0,head_down,C:\Users\LuizAlbertodeAndrade\source\repos\scr...,C:\Users\LuizAlbertodeAndrade\source\repos\scr...
1,domestic_abuse2,strong_signal,1.0000,1.0,1.0,hand_on_face,C:\Users\LuizAlbertodeAndrade\source\repos\scr...,C:\Users\LuizAlbertodeAndrade\source\repos\scr...
2,sad_woman1,strong_signal,1.0000,1.0,1.0,head_down,C:\Users\LuizAlbertodeAndrade\source\repos\scr...,C:\Users\LuizAlbertodeAndrade\source\repos\scr...
3,sad_woman2,strong_signal,0.9565,1.0,1.0,forward_head,C:\Users\LuizAlbertodeAndrade\source\repos\scr...,C:\Users\LuizAlbertodeAndrade\source\repos\scr...


## 13) Inspeção rápida dos artefatos gerados

Esta etapa ajuda a conferir o primeiro JSON v2 disponível e localizar o MP4 anotado correspondente.


In [15]:
# Carrega um JSON v2 já gerado e exibe um resumo mínimo para inspeção rápida.
def load_payload(json_path: str | Path) -> dict[str, Any]:
    return json.loads(Path(json_path).read_text(encoding='utf-8'))


if RUN_RESULTS:
    example_payload = RUN_RESULTS[0]
elif list(OUTPUT_JSON_DIR.glob('*.signals.json')):
    example_payload = load_payload(sorted(OUTPUT_JSON_DIR.glob('*.signals.json'))[0])
else:
    example_payload = None

if example_payload is None:
    display(Markdown('Run the batch cell above to generate JSON and annotated MP4 v2 outputs.'))
else:
    display(Markdown('\n'.join([
        '### Example v2 result',
        f"- Video: `{example_payload['video']['source_name']}`",
        f"- Video level: `{example_payload['video_summary']['level']}`",
        f"- Explanation: `{example_payload['video_summary']['explanation']}`",
        f"- JSON path: `{OUTPUT_JSON_DIR / (example_payload['video']['video_id'] + '.signals.json')}`",
        f"- Annotated video path: `{example_payload['video']['annotated_video_path']}`",
    ])))
    example_payload['video_summary']


### Example v2 result
- Video: `domestic_abuse1.mp4`
- Video level: `strong_signal`
- Explanation: `head down / head tilt is the most frequent trigger`
- JSON path: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\posture_timelines_v2\domestic_abuse1.signals.json`
- Annotated video path: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\annotated_videos_v2\domestic_abuse1.1782764268027.annotated.mp4`